<a href="https://colab.research.google.com/github/rathodgayathri9-glitch/neural-networks-machine-learning-exercises/blob/main/Exercise-08-SOM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
"""Exercise 8 - Self-Organizing Map (SOM) Customer Segmentation and Cluster Exploration
Dataset : Mall Customers (Age, Annual Income (k$), Spending Score (1-100))

* Features are min-max normalised before training.
* BMU (Best Matching Unit) = neuron with the smallest Euclidean distance to the input.
* SOM : the BMU *and its neighbours* on a 1-D neuron chain move toward the input
        (Gaussian neighbourhood, learning rate and radius decay over time).
* CLN : Competitive Learning Network = winner-take-all; only the BMU moves.
Each neuron is one cluster; cluster k = neuron k.

Usage:  python som_customer_segmentation.py [path/to/Mall_Customers.csv]
If no CSV is found a SYNTHETIC stand-in with the same columns is generated (flagged).
"""
import math
import os
import sys

import numpy as np

SEED = 42
FEATS = ["Age", "Annual Income (k$)", "Spending Score"]
DEF_LR, DEF_ITERS, DEF_N = 0.5, 3000, 4


# ----------------------------------------------------------------------------
# Data
# ----------------------------------------------------------------------------
def make_synthetic_mall(seed=1):
    rng = np.random.default_rng(seed)
    # five blobs mimicking the structure of the real Mall Customers data
    spec = [  # n, age(mu,sd), income(mu,sd), spending(mu,sd)
        (39, (25, 5), (26, 7), (78, 10)),
        (23, (45, 11), (26, 7), (21, 10)),
        (80, (43, 14), (55, 8), (50, 8)),
        (35, (41, 11), (88, 16), (17, 10)),
        (23, (32, 4), (86, 16), (82, 8)),
    ]
    rows = []
    for n, a, i, s in spec:
        rows.append(np.column_stack([rng.normal(*a, n), rng.normal(*i, n), rng.normal(*s, n)]))
    X = np.vstack(rows)
    X[:, 0] = np.clip(np.rint(X[:, 0]), 18, 70)
    X[:, 1] = np.clip(np.rint(X[:, 1]), 15, 137)
    X[:, 2] = np.clip(np.rint(X[:, 2]), 1, 99)
    X = X[rng.permutation(len(X))]
    ids = np.arange(1, len(X) + 1)
    return ids, X


def load_mall(path=None):
    cands = ([path] if path else []) + ["Mall_Customers.csv", "mall_customers.csv", "Mall Customers.csv"]
    for p in cands:
        if p and os.path.exists(p):
            import csv
            with open(p, newline="") as f:
                rows = list(csv.reader(f))
            head = [h.strip() for h in rows[0]]
            def col(*names):
                for n in names:
                    for i, h in enumerate(head):
                        if h.lower() == n.lower():
                            return i
                raise KeyError(names)
            i_id = col("CustomerID", "Customer ID")
            i_age = col("Age")
            i_inc = col("Annual Income (k$)", "Annual_Income_(k$)", "Annual Income")
            i_sp = col("Spending Score (1-100)", "Spending_Score", "Spending Score")
            ids = np.array([int(r[i_id]) for r in rows[1:] if r])
            X = np.array([[float(r[i_age]), float(r[i_inc]), float(r[i_sp])] for r in rows[1:] if r])
            return ids, X, p
    print("\n" + "!" * 78)
    print("! WARNING: Mall_Customers.csv not found -> using SYNTHETIC DEMO DATA (200 customers)")
    print("! Put Mall_Customers.csv next to this script (or pass its path) for the real data.")
    print("! Results below are NOT real mall-customer results.")
    print("!" * 78)
    ids, X = make_synthetic_mall()
    return ids, X, "SYNTHETIC"


# ----------------------------------------------------------------------------
# SOM / Competitive learning network
# ----------------------------------------------------------------------------
class SOM:
    def __init__(self, n, lr=DEF_LR, iters=DEF_ITERS, competitive=False, seed=SEED):
        self.n, self.lr, self.iters, self.competitive, self.seed = n, lr, iters, competitive, seed
        self.W = None

    def fit(self, X):
        rng = np.random.default_rng(self.seed)
        self.W = X[rng.choice(len(X), self.n, replace=False)].astype(float).copy()
        sigma0 = max(self.n / 2.0, 1.0)
        pos = np.arange(self.n)
        for t in range(self.iters):
            frac = t / self.iters
            lr_t = self.lr * math.exp(-3.0 * frac)
            x = X[rng.integers(len(X))]
            b = int(np.argmin(np.linalg.norm(self.W - x, axis=1)))       # BMU (Euclidean)
            if self.competitive:
                self.W[b] += lr_t * (x - self.W[b])                      # winner-take-all update
            else:
                sigma = max(sigma0 * math.exp(-3.0 * frac), 0.3)
                h = np.exp(-((pos - b) ** 2) / (2 * sigma ** 2))         # neighbourhood function
                self.W += lr_t * h[:, None] * (x - self.W)
        return self

    def distances(self, x):
        return np.linalg.norm(self.W - x, axis=1)

    def bmu(self, X):
        D = np.linalg.norm(X[:, None, :] - self.W[None, :, :], axis=2)
        idx = D.argmin(axis=1)
        return idx, D[np.arange(len(X)), idx]

    def quantization_error(self, X):
        return float(self.bmu(X)[1].mean())


def hr(t):
    print("\n" + "=" * 78 + f"\n{t}\n" + "=" * 78)


def ask(prompt, default, cast=int, lo=None, hi=None):
    try:
        s = input(f"{prompt} [{default}]: ").strip()
    except EOFError:
        return default
    try:
        v = cast(s) if s else default
    except ValueError:
        return default
    if lo is not None:
        v = max(lo, v)
    if hi is not None:
        v = min(hi, v)
    return v


class System:
    def __init__(self, path=None):
        self.ids, self.raw, src = load_mall(path)
        self.lo, self.hi = self.raw.min(0), self.raw.max(0)
        self.X = self.norm(self.raw)
        self.p33, self.p67 = np.percentile(self.raw, 33.3, axis=0), np.percentile(self.raw, 66.7, axis=0)
        self.som = None
        print(f"[data] source = {src} | {len(self.ids)} customers | features: {', '.join(FEATS)} "
              f"(min-max normalised to [0,1])")

    def norm(self, a):
        return (a - self.lo) / (self.hi - self.lo + 1e-12)

    def denorm(self, a):
        return a * (self.hi - self.lo) + self.lo

    def train(self, n, lr=DEF_LR, iters=DEF_ITERS, competitive=False):
        return SOM(n, lr, iters, competitive).fit(self.X)

    def ensure(self):
        if self.som is None:
            print(f"[info] No SOM trained yet -> training a default SOM with {DEF_N} neurons.")
            self.som = self.train(DEF_N)
        return self.som

    def describe(self, proto):
        age, inc, sp = proto
        a = "Young" if age <= self.p33[0] else "Older" if age >= self.p67[0] else "Middle-Aged"
        i = "Low" if inc <= self.p33[1] else "High" if inc >= self.p67[1] else "Moderate"
        s = "Low" if sp <= self.p33[2] else "High" if sp >= self.p67[2] else "Average"
        return f"{a}, {i}-Income Customers with {s} Spending"

    def cluster_sizes(self, som):
        idx, _ = som.bmu(self.X)
        return np.bincount(idx, minlength=som.n), idx

    # --- 1 -----------------------------------------------------------------
    def m1(self):
        hr("1. CUSTOMER SEGMENTATION USING SOM")
        n = ask("Number of SOM neurons", DEF_N, int, 2, 30)
        lr = ask("Initial learning rate", DEF_LR, float, 0.01, 1.0)
        it = ask("Training iterations", DEF_ITERS, int, 100, 100000)
        self.som = self.train(n, lr, it)
        sizes, idx = self.cluster_sizes(self.som)
        print(f"\nTrained SOM: {n} neurons, lr = {lr}, iterations = {it}, "
              f"quantization error = {self.som.quantization_error(self.X):.4f}")
        for k in range(n):
            members = np.where(idx == k)[0]
            proto = self.denorm(self.som.W[k])
            print(f"\nCluster {k+1} (neuron {k+1}) - {len(members)} customers"
                  f"  | prototype Age {proto[0]:.1f}, Income {proto[1]:.1f}k$, Spending {proto[2]:.1f}")
            if not len(members):
                print("   (empty cluster - no customer won this neuron)")
                continue
            print(f"   {'CustID':>6} {'Age':>5} {'Income(k$)':>11} {'Spending':>9}")
            for j in members:
                a, i_, s = self.raw[j]
                print(f"   {self.ids[j]:>6} {a:>5.0f} {i_:>11.0f} {s:>9.0f}")
        print("\nSummary:  " + "   ".join(f"C{k+1}: {sizes[k]}" for k in range(n)))

    # --- 2 -----------------------------------------------------------------
    def find_customer(self):
        cid = ask(f"Customer ID ({self.ids.min()}-{self.ids.max()})", int(self.ids[0]), int)
        w = np.where(self.ids == cid)[0]
        if not len(w):
            print(f"Customer {cid} not found -> using customer {self.ids[0]}.")
            return 0
        return int(w[0])

    def m2(self):
        hr("2. BEST MATCHING UNIT EXPLORER")
        som = self.ensure()
        j = self.find_customer()
        x = self.X[j]
        d = som.distances(x)
        b = int(d.argmin())
        a, i_, s = self.raw[j]
        print(f"\nCustomer {self.ids[j]}: Age {a:.0f}, Annual Income {i_:.0f}k$, Spending Score {s:.0f}")
        print(f"Normalised input vector: [{x[0]:.3f}, {x[1]:.3f}, {x[2]:.3f}]\n")
        print(f"{'Neuron':>6} {'Weight vector (normalised)':>30} {'Euclidean dist':>15}")
        for k in range(som.n):
            w = som.W[k]
            print(f"{k+1:>6} {'['+', '.join(f'{v:.3f}' for v in w)+']':>30} {d[k]:>15.4f}"
                  f"{'   <-- BMU' if k == b else ''}")
        proto = self.denorm(som.W[b])
        print(f"\nBest Matching Unit : neuron {b+1} (distance {d[b]:.4f})")
        print(f"Winning cluster    : Cluster {b+1} - {self.describe(proto)}")

    # --- 3 -----------------------------------------------------------------
    def imbalance(self, sizes):
        ideal = len(self.ids) / len(sizes)
        big = [k for k, s in enumerate(sizes) if s > 1.5 * ideal]
        small = [k for k, s in enumerate(sizes) if 0 < s < 0.5 * ideal]
        return big, small

    def m3(self):
        hr("3. DYNAMIC CLUSTER COUNT CHALLENGE")
        print("Rules: empty = 0 customers; imbalanced = cluster larger than 1.5x or smaller than "
              "0.5x the ideal share (N/neurons).\n")
        res = []
        for n in (2, 3, 4, 5, 6):
            som = self.train(n)
            sizes, _ = self.cluster_sizes(som)
            empty = [k for k in range(n) if sizes[k] == 0]
            big, small = self.imbalance(sizes)
            qe = som.quantization_error(self.X)
            res.append(dict(n=n, sizes=sizes, empty=empty, big=big, small=small, qe=qe))
            print(f"{n} neurons | sizes: " + ", ".join(f"C{k+1}={sizes[k]}" for k in range(n)))
            print(f"           empty clusters: {[k+1 for k in empty] or 'none'} | "
                  f"too large: {[k+1 for k in big] or 'none'} | too small: {[k+1 for k in small] or 'none'} | "
                  f"QE = {qe:.4f}")
        valid = [r for r in res if not r["empty"] and not r["big"] and not r["small"]]
        pool = valid or [r for r in res if not r["empty"]] or res
        pick, reason = pool[-1], "no elbow found, so the balanced configuration with the lowest QE"
        for a, b in zip(pool, pool[1:]):
            if b["n"] - a["n"] == 1 and (a["qe"] - b["qe"]) / a["qe"] < 0.10:
                pick = a
                reason = "smallest balanced configuration after which adding a neuron improves QE by < 10%"
                break
        print("\nQE per neuron count: " + "  ".join(f"{r['n']}: {r['qe']:.4f}" for r in res))
        print(f"Well-balanced configurations (no empty, none imbalanced): "
              f"{[r['n'] for r in valid] or 'none'}")
        print(f"\nMost suitable number of neurons: {pick['n']}  (QE {pick['qe']:.4f}; {reason})")

    # --- 4 -----------------------------------------------------------------
    def bmu_of(self, row_norm):
        return int(np.argmin(self.som.distances(row_norm)))

    def m4(self):
        hr("4. CUSTOMER CLUSTER MIGRATION ANALYZER")
        som = self.ensure()
        j = self.find_customer()
        f = ask("Feature to modify (1 = Annual Income, 2 = Spending Score)", 2, int, 1, 2)
        fi = f
        name = FEATS[fi]
        base_raw = self.raw[j].copy()
        base = self.bmu_of(self.norm(base_raw))
        print(f"\nCustomer {self.ids[j]}: Age {base_raw[0]:.0f}, Income {base_raw[1]:.0f}k$, "
              f"Spending {base_raw[2]:.0f} -> currently Cluster {base+1}")
        lo_v, hi_v = float(self.lo[fi]), float(self.hi[fi])

        def cluster_at(v):
            r = base_raw.copy()
            r[fi] = v
            return self.bmu_of(self.norm(r))

        def refine(a, b):                       # bisection on [a,b] where the BMU changes
            ca = cluster_at(a)
            for _ in range(30):
                m = (a + b) / 2
                if cluster_at(m) == ca:
                    a = m
                else:
                    b = m
            return b

        for label, step in (("INCREASING", +1.0), ("DECREASING", -1.0)):
            print(f"\n--- {name} {label} (1-unit steps, every 5th step shown + every migration) ---")
            v, cur, k = base_raw[fi], base, 0
            migrations = []
            while lo_v <= v + step <= hi_v:
                prev_v, v = v, v + step
                c = cluster_at(v)
                k += 1
                if c != cur:
                    exact = refine(prev_v, v)
                    migrations.append((exact, cur, c))
                    print(f"  {name} = {v:>6.1f} -> BMU neuron {c+1} (Cluster {c+1})   "
                          f"*** MIGRATION Cluster {cur+1} -> {c+1} at {name} = {exact:.2f} ***")
                    cur = c
                elif k % 5 == 0:
                    print(f"  {name} = {v:>6.1f} -> BMU neuron {c+1} (Cluster {c+1})")
            if migrations:
                for ex, a, b in migrations:
                    print(f"  => Customer migrates from Cluster {a+1} to Cluster {b+1} "
                          f"when {name} {'reaches' if step > 0 else 'falls to'} {ex:.2f}")
            else:
                print(f"  => No migration: customer stays in Cluster {base+1} across the whole range.")

    # --- 5 -----------------------------------------------------------------
    def m5(self):
        hr("5. CLUSTER PROTOTYPE GENERATOR")
        som = self.ensure()
        sizes, _ = self.cluster_sizes(som)
        print(f"Descriptions use dataset terciles (Age {self.p33[0]:.0f}/{self.p67[0]:.0f}, "
              f"Income {self.p33[1]:.0f}/{self.p67[1]:.0f}k$, Spending {self.p33[2]:.0f}/{self.p67[2]:.0f}) "
              f"applied to each learned prototype.\n")
        print(f"{'Cluster':>7} {'Size':>5} {'Age':>7} {'Income(k$)':>11} {'Spending':>9}")
        for k in range(som.n):
            p = self.denorm(som.W[k])
            print(f"{k+1:>7} {sizes[k]:>5} {p[0]:>7.1f} {p[1]:>11.1f} {p[2]:>9.1f}")
        print()
        for k in range(som.n):
            p = self.denorm(som.W[k])
            extra = "   (empty cluster)" if sizes[k] == 0 else ""
            print(f"Cluster {k+1}: {self.describe(p)}{extra}")

    # --- 6 -----------------------------------------------------------------
    def m6(self):
        hr("6. OUTLIER CUSTOMER DETECTOR")
        som = self.ensure()
        idx, dist = som.bmu(self.X)
        thr = dist.mean() + 2 * dist.std()
        order = np.argsort(-dist)
        print(f"BMU distance (normalised space): mean {dist.mean():.4f}, std {dist.std():.4f}; "
              f"outlier threshold = mean + 2*std = {thr:.4f}\n")
        flagged = [j for j in order if dist[j] > thr]
        print(f"Customers above threshold (potential outliers): {len(flagged)}")
        if flagged:
            print(f"{'CustID':>6} {'Age':>5} {'Income':>7} {'Spend':>6} {'Cluster':>8} {'BMU dist':>9}")
            for j in flagged:
                print(f"{self.ids[j]:>6} {self.raw[j,0]:>5.0f} {self.raw[j,1]:>7.0f} {self.raw[j,2]:>6.0f} "
                      f"{idx[j]+1:>8} {dist[j]:>9.4f}")
        print("\nTop 5 most unusual customers:")
        print(f"{'Rank':>4} {'CustID':>6} {'Age':>5} {'Income':>7} {'Spend':>6} {'Cluster':>8} {'BMU dist':>9}")
        for r, j in enumerate(order[:5], 1):
            print(f"{r:>4} {self.ids[j]:>6} {self.raw[j,0]:>5.0f} {self.raw[j,1]:>7.0f} {self.raw[j,2]:>6.0f} "
                  f"{idx[j]+1:>8} {dist[j]:>9.4f}")

    # --- 7 -----------------------------------------------------------------
    def m7(self):
        hr("7. SOM CLUSTER QUALITY COMPARISON")
        iters = ask("Training iterations", DEF_ITERS, int, 100, 100000)
        rows = []
        for model, comp in (("SOM", False), ("CLN", True)):
            for n in (2, 3, 4, 5, 6):
                for lr in (0.1, 0.3, 0.5):
                    som = self.train(n, lr, iters, comp)
                    sizes, _ = self.cluster_sizes(som)
                    rows.append(dict(model=model, n=n, lr=lr, it=iters,
                                     empty=int((sizes == 0).sum()), qe=som.quantization_error(self.X),
                                     big=int(sizes.max()), small=int(sizes.min())))
        print(f"\n{'Model':<5} {'Neurons':>7} {'LR':>5} {'Iters':>6} {'Empty':>6} {'QE':>8} "
              f"{'Largest':>8} {'Smallest':>9}")
        print("-" * 62)
        for r in rows:
            print(f"{r['model']:<5} {r['n']:>7} {r['lr']:>5} {r['it']:>6} {r['empty']:>6} {r['qe']:>8.4f} "
                  f"{r['big']:>8} {r['small']:>9}")
        print("-" * 62)
        ok = [r for r in rows if r["empty"] == 0]
        for n in (3, 4, 5):
            c = [r for r in ok if r["n"] == n]
            if c:
                b = min(c, key=lambda r: r["qe"])
                print(f"Best {n}-neuron configuration (no empty clusters): {b['model']} lr={b['lr']} "
                      f"QE={b['qe']:.4f}, sizes {b['small']}..{b['big']}")
        for m in ("SOM", "CLN"):
            q = [r["qe"] for r in rows if r["model"] == m]
            print(f"Average QE {m}: {np.mean(q):.4f}   empty clusters in total: "
                  f"{sum(r['empty'] for r in rows if r['model']==m)}")
        print("Note: QE always falls as neurons are added, so compare QE only together with "
              "empty/imbalanced clusters (see option 3).")


def main():
    s = System(sys.argv[1] if len(sys.argv) > 1 else None)
    actions = {"1": s.m1, "2": s.m2, "3": s.m3, "4": s.m4, "5": s.m5, "6": s.m6, "7": s.m7}
    while True:
        print("\n" + "=" * 78)
        print(" SOM CUSTOMER SEGMENTATION & CLUSTER EXPLORATION SYSTEM")
        print("=" * 78)
        for n, t in enumerate(["Customer Segmentation using SOM", "Best Matching Unit Explorer",
                               "Dynamic Cluster Count Challenge", "Customer Cluster Migration Analyzer",
                               "Cluster Prototype Generator", "Outlier Customer Detector",
                               "SOM Cluster Quality Comparison", "Exit"], 1):
            print(f" {n}. {t}")
        try:
            ch = input("Enter your choice (1-8): ").strip()
        except EOFError:
            break
        if ch == "8":
            print("Goodbye!")
            break
        if ch in actions:
            actions[ch]()
        else:
            print("Invalid choice. Please enter a number from 1 to 8.")


if __name__ == "__main__":
    main()


!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!
! WARNING: Mall_Customers.csv not found -> using SYNTHETIC DEMO DATA (200 customers)
! Put Mall_Customers.csv next to this script (or pass its path) for the real data.
! Results below are NOT real mall-customer results.
!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!
[data] source = SYNTHETIC | 200 customers | features: Age, Annual Income (k$), Spending Score (min-max normalised to [0,1])

 SOM CUSTOMER SEGMENTATION & CLUSTER EXPLORATION SYSTEM
 1. Customer Segmentation using SOM
 2. Best Matching Unit Explorer
 3. Dynamic Cluster Count Challenge
 4. Customer Cluster Migration Analyzer
 5. Cluster Prototype Generator
 6. Outlier Customer Detector
 7. SOM Cluster Quality Comparison
 8. Exit
Enter your choice (1-8): 1

1. CUSTOMER SEGMENTATION USING SOM
Number of SOM neurons [4]: 2
Initial learning rate [0.5]: 0.2
Training iterations [3000]: 200

Trained SOM: 2 neurons, lr = 0.2